# Guided Lab: Prompt Engineering for LLMs and Generative AI

This notebook is designed as a guided lab for students. You will explore prompt design, compare prompt strategies, and practice reasoning patterns that are widely used in LLM workflows.

## Learning Goals
- Understand in-context learning with zero-shot and few-shot prompting
- Compare outputs from different prompt styles
- Build prompt chaining workflows
- Practice chain-of-thought prompting
- Explore self-consistency and tree-of-thought reasoning

## How to use this notebook
- Read every explanation carefully.
- Complete each TODO block yourself.
- Run the code cells in order.
- Write short observations in the markdown reflection cells.

In [1]:
# Google Colab Installation
# Run this cell if running on Google Colab to install all required dependencies
!pip install -q requests python-dotenv

## Part 1: Prompt Engineering Foundations

A prompt is the instruction or context that guides the model. Small changes in wording, formatting, or examples can produce very different outputs.

### Example to study
Direct prompt example:
- Prompt: Classify the sentiment of the sentence: "I love this product."

Structured prompt example:
- Prompt: You are a sentiment-analysis assistant. Return ONLY one token: positive, negative, or neutral. Sentence: "I love this product."

### Exercise Goal
Compare the answers produced by two different prompts for the same task.

### TODO
Create two prompts for a simple question such as: "Classify the sentiment of the sentence: 'I love this product.'" Use one prompt that is direct and one prompt that includes explicit formatting requirements.

In [5]:
# Optional setup for OpenRouter
import os
import requests

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_API_KEY_HERE")

#OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "deepseek/deepseek-r1:free")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "gpt-4o-mini")


def call_openrouter(prompt, model=OPENROUTER_MODEL, api_key=OPENROUTER_API_KEY):
    if api_key == "YOUR_API_KEY_HERE":
        raise ValueError("Set OPENROUTER_API_KEY in your environment before running the model.")

    response = requests.post(
        "https://openrouter.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print("OpenRouter key configured:", OPENROUTER_API_KEY != "YOUR_API_KEY_HERE")
print("Model:", OPENROUTER_MODEL)

OpenRouter key configured: True
Model: gpt-4o-mini


In [7]:
# TODO: Write your own prompt comparison exercise.
# Complete the two prompts below yourself.
# TODO: Write your own prompt comparison exercise.
# Complete the two prompts below yourself.

prompt_1 = "Classify the sentiment of the sentence: 'I love this product.'"

prompt_2 = """You are a sentiment-analysis assistant.
Analyze the target sentence and return ONLY a JSON object with the keys 'sentiment' and 'confidence_score'.
Target sentence: 'I love this product.'"""

prompts = {
    "Prompt 1: simple": prompt_1,
    "Prompt 2: structured": prompt_2,
}

for name, prompt in prompts.items():
    print(f"\n{name}\n{'-' * 40}")
    print(prompt)
    print("\nTODO: Run this prompt against a model and compare the output.")


Prompt 1: simple
----------------------------------------
Classify the sentiment of the sentence: 'I love this product.'

TODO: Run this prompt against a model and compare the output.

Prompt 2: structured
----------------------------------------
You are a sentiment-analysis assistant. 
Analyze the target sentence and return ONLY a JSON object with the keys 'sentiment' and 'confidence_score'. 
Target sentence: 'I love this product.'

TODO: Run this prompt against a model and compare the output.


## Part 2: In-Context Learning

In-context learning means the model learns from examples or instructions given directly in the prompt. There are two common settings:
- Zero-shot prompting: the model is asked to perform a task without examples.
- Few-shot prompting: the model is given a few examples before the final task.

### Example to study
Zero-shot example:
- Prompt: Classify the following sentence into spam or not spam: "You have won a free prize! Claim now."

Few-shot example:
- Prompt: Classify the following as spam or not spam.
  Example 1: Input: "Congratulations! You have won a cash reward." Output: spam
  Example 2: Input: "Can we meet tomorrow at 3 pm?" Output: not spam
  Now classify: "You have won a free prize! Claim now."

### Why this matters
Few-shot prompting often improves format compliance and task consistency because the model sees the expected pattern.

### TODO
Write one zero-shot prompt and one few-shot prompt for a text classification task. Then compare the results and note which one is more reliable.

In [8]:
# TODO: Write your own zero-shot and few-shot prompts.

zero_shot_prompt = """Classify the customer review as 'Positive' or 'Negative'.
Review: 'The battery life is terrible, but the camera is okay.'"""

few_shot_prompt = """Classify the customer review as 'Positive' or 'Negative'.

Example 1:
Review: 'Awesome phone! The screen is bright and fast.'
Sentiment: Positive

Example 2:
Review: 'Extremely slow and gets hot quickly.'
Sentiment: Negative

Now classify this review:
Review: 'The battery life is terrible, but the camera is okay.'
Sentiment:"""

print("Zero-shot prompt:")
print(zero_shot_prompt)
print("\nFew-shot prompt:")
print(few_shot_prompt)

# TODO: Run both prompts on the same model.
# TODO: Record and compare the answer that each prompt produced.

Zero-shot prompt:
Classify the customer review as 'Positive' or 'Negative'.
Review: 'The battery life is terrible, but the camera is okay.'

Few-shot prompt:
Classify the customer review as 'Positive' or 'Negative'.

Example 1:
Review: 'Awesome phone! The screen is bright and fast.'
Sentiment: Positive

Example 2:
Review: 'Extremely slow and gets hot quickly.'
Sentiment: Negative

Now classify this review:
Review: 'The battery life is terrible, but the camera is okay.'
Sentiment:


## Part 3: Prompt Chaining

Prompt chaining means breaking a complex task into several simpler prompts and sending the output of one prompt into the next. This is useful when the task has multiple stages.

### Example to study
Example chain:
1. Extract three key facts from a paragraph.
2. Summarize those facts in two sentences.
3. Rewrite the summary for a beginner.

### Example workflow
1. Extract key facts from a paragraph.
2. Summarize those facts.
3. Rewrite the summary into a simple answer for a student.

### TODO
Design a short three-step chain for the topic: 'Explain how a large language model works in simple language.'

In [9]:
# TODO: Build your own prompt chain.
# Write the three prompts yourself.

text = """Large Language Models (LLMs) are AI systems trained on vast amounts of text data.
They use deep learning and neural network architectures, specifically transformers, to predict the next most likely word in a sequence based on context.
Through fine-tuning and Reinforcement Learning from Human Feedback (RLHF), LLMs can follow instructions, summarize text, translate languages, and answer complex questions accurately."""

step_1_prompt = f"""Step 1: Extract 3 to 4 core technical concepts from the following text about Large Language Models.
Text: '{text}'"""

step_2_prompt = """Step 2: Take the extracted core concepts from Step 1 and combine them into a brief, easy-to-read summary paragraph."""

step_3_prompt = """Step 3: Rewrite the summary from Step 2 as a fun, simple analogy (like explaining to a 10-year-old) without using any complex jargon."""

print("Step 1 prompt:")
print(step_1_prompt)
print("\nStep 2 prompt:")
print(step_2_prompt)
print("\nStep 3 prompt:")
print(step_3_prompt)

# TODO: Run the chain and write the intermediate outputs here.

Step 1 prompt:
Step 1: Extract 3 to 4 core technical concepts from the following text about Large Language Models.
Text: 'Large Language Models (LLMs) are AI systems trained on vast amounts of text data. 
They use deep learning and neural network architectures, specifically transformers, to predict the next most likely word in a sequence based on context. 
Through fine-tuning and Reinforcement Learning from Human Feedback (RLHF), LLMs can follow instructions, summarize text, translate languages, and answer complex questions accurately.'

Step 2 prompt:
Step 2: Take the extracted core concepts from Step 1 and combine them into a brief, easy-to-read summary paragraph.

Step 3 prompt:
Step 3: Rewrite the summary from Step 2 as a fun, simple analogy (like explaining to a 10-year-old) without using any complex jargon.


## Part 4: Chain-of-Thought Prompting

Chain-of-thought prompting asks the model to show its reasoning steps before giving the final answer. This often improves performance on multi-step problems.

### Example to study
Example prompt:
- Prompt: Solve this problem step by step. A store sells a notebook for 12 dollars and a pen for 3 dollars. If a student buys 2 notebooks and 4 pens, what is the total cost?

### TODO
Use a simple arithmetic or logic problem and ask the model to show its reasoning step by step, then provide the final answer.

In [10]:
# TODO: Write your own chain-of-thought prompt.
# Create the prompt that asks the model to show reasoning before the final answer.

cot_prompt = """Solve the following problem step by step before providing the final answer:

A bakery sells cupcakes for $4 each and cookies for $2 each.
If Sarah buys 3 cupcakes and 5 cookies, and pays with a $30 bill, how much change should she receive?

Show your step-by-step reasoning, then state the final answer clearly."""

print(cot_prompt)
# TODO: Run the prompt and compare the answer style with a direct prompt.

Solve the following problem step by step before providing the final answer:

A bakery sells cupcakes for $4 each and cookies for $2 each. 
If Sarah buys 3 cupcakes and 5 cookies, and pays with a $30 bill, how much change should she receive?

Show your step-by-step reasoning, then state the final answer clearly.


## Part 5: Self-Consistency

Self-consistency improves reliability by asking the model to generate multiple reasoning paths and then selecting the most common answer. This is useful when a problem has more than one reasonable way to be solved.

### Example to study
Example idea:
- Ask the same time-difference question three times using slightly different wording.
- Compare the final answers and choose the answer that appears most often.

### TODO
Design three versions of the same question and ask the model to answer them independently. Then compare the outputs and identify the answer that appears most often.

In [11]:
# TODO: Write your own self-consistency experiment.
# Create three prompt versions that should answer the same question.

question = "What is the time difference between Cairo and Tokyo?"

versions = [
    f"Direct question: {question} Please state the exact time difference in hours.",
    f"Step-by-step phrasing: If it is 12:00 PM in Cairo, what time is it in Tokyo? Show your calculation and conclude with the total hour difference for: '{question}'",
    f"Role-based phrasing: As a timezone assistant, tell me: {question} Make sure to specify which city is ahead.",
]

for idx, version in enumerate(versions, start=1):
    print(f"Version {idx}:")
    print(version)
    print("\nTODO: Run the model on this version and record its answer.\n")

Version 1:
Direct question: What is the time difference between Cairo and Tokyo? Please state the exact time difference in hours.

TODO: Run the model on this version and record its answer.

Version 2:
Step-by-step phrasing: If it is 12:00 PM in Cairo, what time is it in Tokyo? Show your calculation and conclude with the total hour difference for: 'What is the time difference between Cairo and Tokyo?'

TODO: Run the model on this version and record its answer.

Version 3:
Role-based phrasing: As a timezone assistant, tell me: What is the time difference between Cairo and Tokyo? Make sure to specify which city is ahead.

TODO: Run the model on this version and record its answer.



## Part 6: Tree-of-Thought Reasoning

Tree-of-thought prompting extends the idea of step-by-step reasoning by exploring multiple branches of thought. The model considers different candidate paths, evaluates them, and chooses the best one.

### Example to study
Example decision problem:
- Problem: A student must choose between three study methods for a final exam. Which one is most effective?
- Branch 1: Use flashcards and active recall.
- Branch 2: Read the notes once without any practice.
- Branch 3: Practice with a timed quiz and review mistakes.

### TODO
Create a small decision problem involving at least three possible choices. Show one branch for each choice and explain which branch is the best based on evidence.

In [12]:
# TODO: Design your own tree-of-thought example.
# Write a problem and at least three possible branches.

problem = "A micro-controller system experiences intermittent communication errors over SPI. Which solution branch should the engineering team prioritize?"

branches = [
    "Branch 1: Increase the SPI clock frequency to speed up data transfers.",
    "Branch 2: Add pull-up resistors and place decoupling capacitors near the power pins to reduce electrical noise.",
    "Branch 3: Replace the entire microcontroller chip immediately with a higher-end model.",
]

for branch in branches:
    print(branch)

print("\nTODO: Evaluate each branch and explain why one branch wins.")

Branch 1: Increase the SPI clock frequency to speed up data transfers.
Branch 2: Add pull-up resistors and place decoupling capacitors near the power pins to reduce electrical noise.
Branch 3: Replace the entire microcontroller chip immediately with a higher-end model.

TODO: Evaluate each branch and explain why one branch wins.


## Part 7: Reflection and Comparison

Answer the following questions in your own words:
1. Which prompt style was most consistent in your experiments?

"Few-shot prompting was the most consistent because providing explicit examples guided the model to stick to the required output format"

2. What is the main difference between few-shot and zero-shot prompting?

"Zero-shot asks the model to perform a task without any prior examples, whereas Few-shot includes concrete examples to clarify expected input and output"

3. Why is prompt chaining useful for complex tasks?

"It breaks down a complex task into smaller, manageable steps, allowing us to inspect and refine intermediate results along the way"

4. How does chain-of-thought differ from direct prompting?

"Direct prompting asks for an immediate answer, while Chain-of-Thought forces the model to show its step-by-step reasoning first, which increases accuracy"

5. What is the value of self-consistency and tree-of-thought reasoning?

"Self-consistency verifies reliability by selecting the most repeated answer across multiple runs, while Tree-of-Thought explores and evaluates multiple solution branches to pick the optimal path"


### TODO
Write a short paragraph summarizing what you learned from the lab.

In this lab, I explored how structuring prompts directly influences model behavior. I observed that Few-Shot prompting enforces strict output formats, while Prompt Chaining simplifies complex workflows. Additionally, reasoning frameworks like Chain-of-Thought and Tree-of-Thought significantly improve reliability for multi-step problem-solving.

In [13]:
# Reflection checklist
reflection_notes = [
    "I compared outputs from different prompt styles.",
    "I identified differences between zero-shot and few-shot prompting.",
    "I built a prompt chain and described the intermediate steps.",
    "I explored chain-of-thought, self-consistency, and tree-of-thought.",
]

for note in reflection_notes:
    print(note)

I compared outputs from different prompt styles.
I identified differences between zero-shot and few-shot prompting.
I built a prompt chain and described the intermediate steps.
I explored chain-of-thought, self-consistency, and tree-of-thought.
